In [1]:
import csv
import importlib
import os
import random
import sys
import torch
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from time import sleep
from collections import deque, defaultdict
from itertools import count
from typing import Any, Dict, Counter, List

sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

from importnb import Notebook
with Notebook():
    from Labs.LatencyModel import LatencyModel, MultiDULatencyModel
    from Labs.Policy import DrlPolicy
    from Labs.CacheEngine import CacheEngineEnv
    from Labs.UserRequest import UserRequestEvents
    from Labs.EnvWrapper import EnvWrapper

from RL.Networks import QNetwork, MultiHeadQNetwork
from RL.Buffers import ReplayBuffer, NStepReplayBuffer
from RL.Adapters import FeatureAdapter, NetworkAdapter
from RL.FocusWorkers import BaseWorker, EnhWorker, FocusWorker
from RL.A2CWorker import A2CWorker

import Common.config as config
import Common.datatypes as datatypes
import Common.debugger as debugger
import Common.utils as utils
import Core.builders as builders

importlib.reload(builders)
importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(debugger)
importlib.reload(utils)

<module 'Common.utils' from '/home/eduardo/Workspace/CacheVideoPredict360/Sources/Common/utils.py'>

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

UserTransition = datatypes.UserTransition
CachePolicy = datatypes.CachePolicy
CacheKey = datatypes.CacheKey

cfg = config.Config()
cfg.filename = \
    f"focus_eps{cfg.epsilon_start}_" \
    f"lrdecay{cfg.learning_rate_decay}_" \
    f"gamma{cfg.gamma}.csv"
cfg.state_dim_base_focus = cfg.cache_size * 10 + 2
cfg.state_dim_enh_focus = cfg.cache_size * 10 + 2
cfg.action_dim_base_focus = cfg.cache_size * 5 + 1

debugger = debugger.debug

In [3]:
class NetworkAdapter:
    def __init__(self, cfg: Any, env: Any, feature_adapter: Any):
        self.env = env
        self.cfg = cfg
        self.features = feature_adapter

        self.C = self.cfg.cache_size  # paper's cache capacity (videos)
        self.k = self.cfg.viewport    # paper's tiles per video (enhancement)

    def build_observation(self, video, tile = None) -> np.ndarray:

        cache = self.env.mec_cache.policy.cache

        x_s = np.zeros(len(cache), dtype=np.float32)
        x_l = np.zeros(len(cache), dtype=np.float32)
            
        for idx, (v, t) in enumerate(cache):
            if v == -1:
                continue

            if t == -1:
                x_s[idx] = self.features.video_freq_short.get(v, 0) / self.features.video_hist_short.maxlen
                x_l[idx] = self.features.video_freq_long.get(v, 0) / self.features.video_hist_long.maxlen
            else:
                x_s[idx] = self.features.tile_freq_short.get((v, t), 0) / self.features.tile_hist_short.maxlen
                x_l[idx] = self.features.tile_freq_long.get((v, t), 0) / self.features.tile_hist_long.maxlen

        if tile is None:
            y_s = np.array(
                [self.features.video_freq_short.get(video, 0) / self.features.video_hist_short.maxlen], 
                dtype=np.float32
            )
            y_l = np.array(
                [self.features.video_freq_long.get(video, 0) / self.features.video_hist_long.maxlen], 
                dtype=np.float32
            )
        else:
            y_s = np.array(
                [self.features.tile_freq_short.get((video, tile), 0) / self.features.tile_hist_short.maxlen], 
                dtype=np.float32
            )
            y_l = np.array(
                [self.features.tile_freq_long.get((video, tile), 0) / self.features.tile_hist_long.maxlen], 
                dtype=np.float32
            )

        return np.concatenate([x_s, x_l, y_s, y_l], axis=0)
    
    def build_observation_back(self, req) -> np.ndarray:
        
        if req is None:
            return (
                np.zeros(self.cfg.state_dim_base_focus, dtype=np.float32), 
                np.zeros((self.k, self.cfg.state_dim_enh_focus), dtype=np.float32)
            )

        video = req["video"]
        viewport = req["viewport"]
        
        cache = self.env.mec_cache.policy.cache

        x_s = np.zeros(len(cache), dtype=np.float32)
        x_l = np.zeros(len(cache), dtype=np.float32)

            
        for idx, (v, t) in enumerate(cache):
            if v == -1:
                continue

            if t == -1:
                x_s[idx] = self.features.video_freq_short.get(v, 0) / self.features.video_hist_short.maxlen
                x_l[idx] = self.features.video_freq_long.get(v, 0) / self.features.video_hist_long.maxlen
            else:
                x_s[idx] = self.features.tile_freq_short.get((v, t), 0) / self.features.tile_hist_short.maxlen
                x_l[idx] = self.features.tile_freq_long.get((v, t), 0) / self.features.tile_hist_long.maxlen

        y_s = np.array(
            [self.features.video_freq_short.get(video, 0) / self.features.video_hist_short.maxlen], dtype=np.float32
        )
        y_l = np.array(
            [self.features.video_freq_long.get(video, 0) / self.features.video_hist_long.maxlen], dtype=np.float32
        )

        state_video = np.concatenate([x_s, x_l, y_s, y_l], axis=0) 
        
        state_vp = []
        for i, tile in enumerate(viewport):
            z_s = np.array(
                [self.features.tile_freq_short.get((video, tile), 0) / self.features.tile_hist_short.maxlen], dtype=np.float32
            )
            z_l = np.array(
                [self.features.tile_freq_long.get((video, tile), 0) / self.features.tile_hist_long.maxlen], dtype=np.float32
            )

            state_vp.append(np.concatenate([x_s, x_l, z_s, z_l], axis=0))
            
        return state_video, state_vp

    def reset(self):
        
        obs, info = self.env.reset()
        self.features.reset_history()

        return obs, info
    
    def env_is_done(self) -> bool:
        return self.env.users_env.all_users_done()

In [4]:
def _append_csv_row(csv_path: str, fieldnames: list[str], row: dict) -> None:
    write_header = not os.path.exists(csv_path)
    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(row)


def save_episode_metrics(
    metrics_dir: str,
    ep: int,
    total_reward: float,
    cache_hits: int,
    cache_misses: int,
    agent,
):
    csv_path = os.path.join(metrics_dir, 'episode_metrics.csv')
    fieldnames = [
        'episode',
        'total_reward',
        'cache_hits',
        'cache_misses',
        'hit_rate',
        'epsilon',
        'lr',
    ]

    row = {
        'episode': ep,
        'total_reward': round(float(total_reward), 2),
        'cache_hits': cache_hits,
        'cache_misses': cache_misses,
        'hit_rate': float(cache_hits) / float(cache_hits + cache_misses + 1e-9),
        'epsilon': round(float(agent.epsilon), 6) if agent else None,
        'lr': float(agent.scheduler.get_last_lr()[0]) if agent else None,
    }
    _append_csv_row(csv_path, fieldnames, row)

def save_step_metrics(
    metrics_dir: str,
    episode: int,
    episode_step: int,
    global_step: int,
    reward: float,
    bs_hits: int,
    bs_miss: int,
    e_hits: int,
    e_miss: int,
    agent,
    train_metrics: dict | None,
):
    csv_path = os.path.join(metrics_dir, 'step_metrics.csv')
    fieldnames = [
        'episode',
        'episode_step',
        'global_step',
        'reward',
        'base_hits',
        'base_misses',
        'enh_hits',
        'enh_misses',
        'cache_hits',
        'cache_misses',
        'epsilon',
        'lr',
        'train_loss',
        'actor_loss',
        'critic_loss',
        'advantage_mean',
        'value_mean',
        'target_value_mean',
    ]

    row = {
        'episode': episode,
        'episode_step': episode_step,
        'global_step': global_step,
        'reward': float(reward),
        'base_hits': int(bs_hits),
        'base_misses': int(bs_miss),
        'enh_hits': int(e_hits),
        'enh_misses': int(e_miss),
        'cache_hits': int(bs_hits + e_hits),
        'cache_misses': int(bs_miss + e_miss),
        'epsilon': round(float(agent.epsilon), 6) if agent else None,
        'lr': float(agent.scheduler.get_last_lr()[0]) if agent else None,
        'train_loss': None,
        'actor_loss': None,
        'critic_loss': None,
        'advantage_mean': None,
        'value_mean': None,
        'target_value_mean': None,
    }

    if train_metrics is not None:
        row.update({
            'train_loss': train_metrics.get('train_loss'),
            'actor_loss': train_metrics.get('actor_loss'),
            'critic_loss': train_metrics.get('critic_loss'),
            'advantage_mean': train_metrics.get('advantage_mean'),
            'value_mean': train_metrics.get('value_mean'),
            'target_value_mean': train_metrics.get('target_value_mean'),
        })

    _append_csv_row(csv_path, fieldnames, row)


def update_metrics(info: dict, reward: float) -> tuple[float, int, int, int, int]:
    enh_hits = info.get("enh_layer_hits", 0)
    base_hits = info.get("base_layer_hits", 0)
    enh_misses = info.get("enh_layer_misses", 0)
    base_misses = info.get("base_layer_misses", 0)

    return reward, base_hits, base_misses, enh_hits, enh_misses

In [ ]:
def select_action(agent, req_state, env, net_adapter=None):

    if req_state is None:
        return None, np.zeros(5, dtype=np.int32), [None] * (1 + cfg.viewport)

    backhaul_usage = 0
    missing = env._missing_items(req_state)
    transition = [None] * (1 + cfg.viewport)

    if missing[0] == 1:
        state_base = net_adapter.build_observation(req_state["video"])
        action_base, value_base, prob_base = agent.select_action(state_base)

        transition[0] = {
            'state': state_base,
            'action': action_base,
            'value': value_base,
            'probs': prob_base
        }

        message = {
            "video": req_state["video"],
            "tiles": [],
            "base_req_init": True,
            "action_idx": action_base,
        }
        env.prefetch_fn(env.mec_cache, message)

        if action_base != 0:
            backhaul_usage += 12 * env.mec_cache.tile_size_bytes[0]

    for idx, missing_item in enumerate(missing[1:]):

        if missing_item == 1:
            state_enh = net_adapter.build_observation(
                req_state["video"],
                req_state["viewport"][idx]
            )
            action_enh, value_enh, prob_enh = agent.select_action(state_enh)

            transition[idx + 1] = {
                'state': state_enh,
                'action': action_enh,
                'value': value_enh,
                'probs': prob_enh
            }

            message = {
                "video": req_state["video"],
                "tiles": [req_state["viewport"][idx]],
                "base_req_init": False,
                "action_idx": action_enh,
            }
            env.prefetch_fn(env.mec_cache, message)

            if action_enh != 0:
                backhaul_usage += env.mec_cache.tile_size_bytes[1]

    debugger.log("backhaul_usage", backhaul_usage)

    debugger.log("base_layer_miss", missing[0])
    for i, is_missing in enumerate(missing[1:], start=1):
            debugger.log(f"enh_layer_missing_{i}", is_missing)

    return None, missing, transition


def run_episode(episode, env, agent, net_adapter, cfg, metrics_dir, global_step_start):
    """Run one full training episode and persist step-level metrics."""
    _, info = net_adapter.reset()

    total_reward = 0.0
    cache_hits = cache_misses = 0
    base_hits = base_misses = 0
    enh_hits = enh_misses = 0

    global_step = global_step_start

    if cfg.has_warmup:
        env.warmup_phase(net_adapter, 1000)

    for step in range(cfg.max_steps):
        global_step += 1

        # --- Build State ---
        req_state = info.get("user_request", None)

        # --- Action Selection ---
        action, missing, transition = select_action(agent, req_state, env, net_adapter)

        # --- Environment Step ---
        _, reward, done, info = env.step(action, req_state, net_adapter)

        # --- Store Transition & Train ---
        nxt_req = info["user_request"]

        reward_0 = info["reward_layer_0"]
        reward_1 = info["reward_layer_1"]
        reward = reward_0 + reward_1

        queued_update = False

        if missing[0] == 1 or transition[0] is not None:
            next_state_base = net_adapter.build_observation(nxt_req["video"])

            agent.remember(
                transition[0]['probs'],
                transition[0]['value'],
                reward,
                next_state_base,
                done
            )
            queued_update = True

        for i in range(len(missing) - 1):
            if missing[i + 1] == 1 and transition[i + 1] is not None:
                next_state_enh = net_adapter.build_observation(nxt_req["video"], nxt_req["viewport"][i])
                agent.remember(
                    transition[i + 1]['probs'],
                    transition[i + 1]['value'],
                    reward,
                    next_state_enh,
                    done
                )
                queued_update = True

        train_metrics = None
        if queued_update:
            train_metrics = agent.train_step()

        delta_r, bs_hits, bs_miss, e_hits, e_miss = update_metrics(info, reward)
        total_reward += delta_r
        cache_hits += bs_hits + e_hits
        cache_misses += bs_miss + e_miss
        base_hits += bs_hits
        base_misses += bs_miss
        enh_hits += e_hits
        enh_misses += e_miss

        save_step_metrics(
            metrics_dir=metrics_dir,
            episode=episode,
            episode_step=step,
            global_step=global_step,
            reward=delta_r,
            bs_hits=bs_hits,
            bs_miss=bs_miss,
            e_hits=e_hits,
            e_miss=e_miss,
            agent=agent,
            train_metrics=train_metrics,
        )

        if done:
            break

        debugger.log('cache_hits', bs_hits + e_hits)
        debugger.log('cache_misses', bs_miss + e_miss)

    return total_reward, cache_hits, cache_misses, base_hits, base_misses, enh_hits, enh_misses, global_step


def train(cfg):
    env = builders.build_environment(cfg)

    agent = A2CWorker(cfg, debugger=debugger)

    feature_adapter = FeatureAdapter(cfg, env)
    net_adapter = NetworkAdapter(cfg, env, feature_adapter)

    date_dir = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M")
    debug_path = os.path.join(cfg.path_results, date_dir)
    metrics_dir = os.path.join(debug_path, "metrics")
    os.makedirs(metrics_dir, exist_ok=True)

    print(f"Starting training for {cfg.n_episodes} episodes... {date_dir}")
    print(f"Warmup Phase: {'Enabled' if cfg.has_warmup else 'Disabled'}")
    print(f"Users Session Length: {cfg.user_session_length}")
    print(
        f"State Dim Base: {agent.state_dim}, Action Dim Base: {agent.action_dim}, Hidden Dim Base: {cfg.hidden_dim_base_focus}"
    )

    global_step = 0

    for episode in range(cfg.n_episodes):

        total_reward, hits, misses, bs_hits, bs_miss, enh_hits, enh_miss, global_step = run_episode(
            episode, env, agent, net_adapter, cfg, metrics_dir, global_step
        )

        agent.update_epsilon()

        save_episode_metrics(
            metrics_dir=metrics_dir,
            ep=episode,
            total_reward=total_reward,
            cache_hits=hits,
            cache_misses=misses,
            agent=agent,
        )

        debugger.log('lr', agent.scheduler.get_last_lr()[0])
        debugger.log('epsilon', agent.epsilon)

        print(
            f"--- Episode {episode} | R: {total_reward:.2f} | "
            f"HR: {hits / (hits + misses + 1e-9):.2f} | "
            f"BHR: {bs_hits / (bs_hits + bs_miss + 1e-9):.2f} | "
            f"EHR: {enh_hits / (enh_hits + enh_miss + 1e-9):.2f} ---"
        )

        debugger.save_results(filepath=f"{debug_path}/debug_ep{episode}")
        debugger.clear()

        print("-" * 50)


if __name__ == "__main__":
    train(cfg)

Starting training for 400 episodes... 2026-03-25_18-37
Warmup Phase: Enabled
Users Session Length: 60
State Dim Base: 1252, Action Dim Base: 626, Hidden Dim Base: 512


/home/eduardo/Workspace/CacheVideoPredict360/Sources/RL/A2CWorker.py:92: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  next_state = torch.FloatTensor(next_state).to(self.device)


--- Episode 0 | R: 267847.50 | HR: 0.62 | BHR: 0.70 | EHR: 0.37 ---
--------------------------------------------------
--- Episode 1 | R: 290810.00 | HR: 0.67 | BHR: 0.81 | EHR: 0.28 ---
--------------------------------------------------
--- Episode 2 | R: 287912.50 | HR: 0.67 | BHR: 0.80 | EHR: 0.25 ---
--------------------------------------------------
--- Episode 3 | R: 300047.50 | HR: 0.69 | BHR: 0.83 | EHR: 0.29 ---
--------------------------------------------------
--- Episode 4 | R: 305195.00 | HR: 0.71 | BHR: 0.85 | EHR: 0.28 ---
--------------------------------------------------
--- Episode 5 | R: 300590.00 | HR: 0.70 | BHR: 0.84 | EHR: 0.27 ---
--------------------------------------------------
--- Episode 6 | R: 303032.50 | HR: 0.70 | BHR: 0.85 | EHR: 0.27 ---
--------------------------------------------------
--- Episode 7 | R: 298130.00 | HR: 0.69 | BHR: 0.83 | EHR: 0.28 ---
--------------------------------------------------


KeyboardInterrupt: 